# Tecan Carousel: cartridges, transfers and barcodes

```{device-card} tecan-carousel
```

This guide is the working walkthrough for a Tecan Carousel LPT220 EVO: how to describe the cassettes
that are actually installed, how to move a plate from a cartridge slot to the transfer station, and
how to read plate barcodes in place. It is written against the unit in the DALSA lab, so the measured
numbers below are that machine's, not general constants — re-measure them for a different unit.

Everything that moves is marked **MOTION**. Nothing in this notebook runs by itself; every hardware
cell needs an operator present and a clear path.


## What the numbers mean

Two coordinate systems meet here and confusing them is the main source of wrong-plate mistakes.

- **Slot** — a physical shelf in the cassette, counted **from the top**: slot 1 is the topmost plate.
  This is how the lab talks about plates and how you should talk about them too.
- **Level** — the controller's address along the lift stroke. Every operation in the API takes a
  level, so translate with `level_for_slot()` and never hand-count.

Both the number of slots and the level numbering depend on the cassette fitted:

| Cassette | Slots | Levels per slot | Slot 1 (top) | Slot 7 (bottom) |
|---|---|---|---|---|
| Standard 22-slot | 22 | 1 | level 1 | level 22 |
| Deep well 7-slot | 7 | 3 | level 22 | level 4 |

Two details of the deep-well cassette are easy to get wrong and were both measured on this unit:

- The top slot is **level 22**, not level 19. A cassette that starts at level 1 gives a mapping one
  whole slot too low, and every scan misses.
- Levels 1-3 are below the cassette. They are valid lift positions but they are not slots.


## Before you start

Check these with the operator before the first command:

1. The serial adapter is the right one and someone has read/write access to the port
   (`ls -l /dev/serial/by-id/`), and no other program holds it.
2. The **shovel is empty** and the **transfer station is empty**. Most operations refuse to start
   otherwise, and that refusal is deliberate.
3. The cassette map matches what is physically installed — slots, levels per slot, and the top level.
4. Barcode reads additionally move `DM27` (the reader's height offset) and put the handler into
   barcode positioning mode. Both are restored, but a killed process may leave them set; see
   *Troubleshooting*.


## 1. Describe the installed cassettes

Build the device with the cassette geometry. `cassette_slots` and `cassette_level_step` describe the
cassette; `cassette_levels_increase_downward` says which way the controller counts; and
`cassette_top_level` pins the cassette's top slot when it is not where the geometry alone would put
it.

The configuration below is this unit: cartridges 1-8 hold 7-slot deep-well cassettes, cartridges 9-10
hold standard 22-slot cassettes, and cartridge 1's deep well was measured with its top slot at level
22.

In [ ]:
# **HARDWARE: do not execute without per-run operator approval.**
from pylabrobot.tecan import BarcodeRead, TecanCarousel

PORT = "/dev/serial/by-id/usb-FTDI_FT232R_USB_UART_B003626H-if00-port0"

carousel = TecanCarousel(
  port=PORT,
  number_of_cartridges=10,
  levels_per_cartridge=22,
  cassette_slots={**{n: 7 for n in range(1, 9)}, 9: 22, 10: 22},
  cassette_level_step={**{n: 3 for n in range(1, 9)}, 9: 1, 10: 1},
  # Deep-well cassettes count controller levels upward from the bottom; standard cassettes count
  # downward from the top. Public positions are top-first either way.
  cassette_levels_increase_downward={**{n: False for n in range(1, 9)}, 9: True, 10: True},
  # Measured on cartridge 1: its top slot is level 22, so slot 7 is level 4. The other deep-well
  # cartridges were not measured; drop this entry if they are mounted the ordinary way.
  cassette_top_level={1: 22},
)

Print the map once and check it against the cassette in front of you. It costs nothing and prevents
the most common failure in this notebook: addressing a plate one slot away from the one you meant.

In [ ]:
for cartridge, slots in ((1, 7), (9, 22)):
  mapping = {slot: carousel.level_for_slot(cartridge, slot) for slot in range(1, slots + 1)}
  print(f"cartridge {cartridge}: {slots} slots -> {mapping}")

The translation is a bijection, so a level always maps back to exactly one slot. Levels that fall
between two deep-well slots raise instead of guessing.

In [ ]:
for cartridge, slots in ((1, 7), (9, 22)):
  for slot in range(1, slots + 1):
    level = carousel.level_for_slot(cartridge, slot)
    assert carousel.slot_for_level(cartridge, level) == slot

# A level that is not a slot on a three-levels-per-slot cassette:
try:
  carousel.slot_for_level(1, 20)
except ValueError as error:
  print("expected:", error)

## 2. Connect, then check the state

`setup()` opens the link and completes the `CR`/`CC` handshake. It does not home or move anything.
`initialize()` is the one that activates and homes the handling (`ST 1801`), and it is idempotent: if
the handling is already active it sends nothing.

In [ ]:
# **HARDWARE: do not execute without per-run operator approval.**
await carousel.setup()
print(await carousel.get_status())

In [ ]:
# **HARDWARE: do not execute without per-run operator approval.**
status = await carousel.get_status()
if not status.active:
  await carousel.initialize()          # MOTION: homes the carousel and lift
print("ready:", status.ready, "| error:", status.error, "| active:", status.active)

## 3. Move a plate from a slot to the transfer station

`export_plate(cartridge, level)` rotates to the cartridge, lifts to the level, picks the plate onto the
shovel and places it on the transfer station (`ST 1905`). It refuses to start if the shovel or the
station is already occupied, and after it reports ready it checks the sensors rather than trusting the
acknowledgement.

In [ ]:
# **HARDWARE: do not execute without per-run operator approval.**
# MOTION
cartridge, slot = 9, 21
await carousel.export_plate(cartridge, carousel.level_for_slot(cartridge, slot))
print("plate is on the transfer station")

Putting it back is `import_plate(cartridge, level)` (`ST 1904`): the plate on the station is picked up
and stored at the requested slot. Use it to undo an export.

In [ ]:
# **HARDWARE: do not execute without per-run operator approval.**
# MOTION
await carousel.import_plate(cartridge, carousel.level_for_slot(cartridge, slot))
print("plate is back in the cassette")

### Several cartridges in one run

Loop over a plan rather than calling the same line by hand. Each entry is `(cartridge, slot)`, slots
counted from the top. Stop on the first failure: a failed handling operation faults the session, and
`stop()` followed by `setup()` is what clears it.

In [ ]:
# **HARDWARE: do not execute without per-run operator approval.**
PLAN = [(9, 2), (9, 21), (1, 1), (1, 7)]

for cartridge, slot in PLAN:
  level = carousel.level_for_slot(cartridge, slot)
  print(f"exporting cartridge {cartridge} slot {slot} (level {level})")
  await carousel.export_plate(cartridge, level)
  print("  on the transfer station")

  await carousel.import_plate(cartridge, level)
  print("  stored again")

## 4. Read plate barcodes in place

`read_barcode(cartridge, level)` reads the plate without exporting it. It switches the reader on
(`ST 1712`), enables two-way communication (`ST 1200`), puts the lift into barcode positioning
(`ST 1910`), scans (`ST 1710`) and then leaves the mode again (`ST 1713`, `RS 1200`, `RS 1910`).

The result is a `BarcodeRead`:

| Field | Meaning |
|---|---|
| `barcode` | the barcode without its Code128 check character, or `None` when nothing was read |
| `payload` | the raw text the reader sent, including the check character (`000VJ86"`) |
| `position` | the reader's own `SPS:` field for the position it scanned |
| `aim_delta` | the aim offset, in controller units, where the barcode was found |

Reading is **not** a single scan. The reader's vertical window is narrow and the labels do not sit
exactly at the nominal read height, so the method sweeps a small band of aim offsets — by default 0,
50, 100, 150 and 200 units above the slot's nominal position — and stops at the first position that
returns a payload. That is why a read takes 10-30 seconds rather than one second.

In [ ]:
# **HARDWARE: do not execute without per-run operator approval.**
scan = await carousel.read_barcode(9, carousel.level_for_slot(9, 21))
print(f"barcode={scan.barcode!r} payload={scan.payload!r} position={scan.position!r} aim={scan.aim_delta:+d}")

An empty slot is not an error. The reader reports `??`, which the driver turns into
`barcode=None`: the slot is empty, or the plate in it has no readable label. Compare that with an
exception, which means the exchange itself went wrong.

In [ ]:
# **HARDWARE: do not execute without per-run operator approval.**
empty = await carousel.read_barcode(9, carousel.level_for_slot(9, 1))
print("slot 1:", "nothing read" if empty.barcode is None else empty.barcode)

Scanning a whole cartridge: read each occupied slot, keep the barcodes, and treat `None` as
"unknown", never as "empty" without a second look.

In [ ]:
# **HARDWARE: do not execute without per-run operator approval.**
OCCUPIED = {9: [2, 11, 21, 22], 1: [1, 4, 7]}

barcodes = {}
for cartridge, slots in OCCUPIED.items():
  for slot in slots:
    level = carousel.level_for_slot(cartridge, slot)
    result = await carousel.read_barcode(cartridge, level)
    barcodes[(cartridge, slot)] = result.barcode
    print(f"cartridge {cartridge} slot {slot} (level {level}): {result.barcode or '??'}")

### Trade-off: speed

The default sweep is what makes a marginal label readable. When you know the plates are seated well
and you want speed, pass a single aim:

```python
fast = await carousel.read_barcode(9, 21, sweep=(0,))
```

That is one scan instead of up to five, at the cost of returning `??` for a plate whose label is a
millimetre off centre. Keep the default for unknown plates.

## 5. Worked recipe: identify first, then move

The useful combination is to read the cassette, decide which plate you want by barcode, and only then
export it. The transfer station holds one plate, so the decision has to happen before anything moves.

In [ ]:
# **HARDWARE: do not execute without per-run operator approval.**
WANTED = "000VJ86"

found = None
for cartridge, slots in OCCUPIED.items():
  for slot in slots:
    result = await carousel.read_barcode(cartridge, carousel.level_for_slot(cartridge, slot))
    if result.barcode == WANTED:
      found = (cartridge, slot)
      break
  if found:
    break

print("wanted plate is at", found)
# MOTION: uncomment when an operator is ready. export_plate takes a level, not a slot.
# if found:
#   await carousel.export_plate(found[0], carousel.level_for_slot(*found))

## 6. Finish cleanly

`stop()` sends `CQ` to end the controller conversation and closes the port. It is not a physical stop;
if a plate is on the shovel or the station, it stays there. If a barcode read was interrupted, `stop()`
also tries to leave barcode positioning mode before closing.

In [ ]:
# **HARDWARE: do not execute without per-run operator approval.**
await carousel.stop()

## Troubleshooting

| Symptom | Meaning and what to do |
|---|---|
| `??` on every aim | Nothing readable at that slot: plate missing, label damaged, or the slot/level map is wrong. Check the map against the cassette before retrying. |
| `ValueError: level must be an integer in 1..N` | The geometry rejects the address. For a deep-well cassette this usually means `cassette_top_level` is missing. |
| `CarouselError: controller has an error, DM200=...` | A latched handling error. Clear it with `reset()` (re-homes) or `soft_reset()`, then `initialize()`. Do not treat either as a retry. |
| `... could not be confirmed out of barcode positioning mode` | The driver could not prove it left `ST 1910`. Stop, inspect the handler with the operator, and only continue after a status read shows a sane state. |
| `RuntimeError: Faulted session` | The previous operation failed or was interrupted. `stop()` then `setup()`; confirm the physical state first. |
| A killed script leaves `DM27` changed | `DM27` is the reader height offset, restored after every read but not after a hard kill. Read it (`RD DM27`), write back the documented value, verify the readback. |

Notes that matter for this unit but are not general truths:

- `DM200` reads 16384 at rest here, so a 16384 reading alone does not prove the handler is in barcode
  mode. The driver only fails a read when the value changed from what it saw at entry.
- The barcode reader needs `DM27 = 0` for the labels on this machine. The delivered configuration uses
  550, which aims about one plate pitch low; the driver does not change that register permanently.


## Where to go next

- `hello-world.ipynb` in this folder covers the full operation set: positioning, internal moves,
  shovel recovery, soft reset.
- The API reference is `pylabrobot.tecan.TecanCarousel`, and `BarcodeRead` documents the scan result.
- Any change to the cassette inventory belongs in the constructor arguments, not in ad-hoc level
  arithmetic in a script.
